<a href="https://colab.research.google.com/github/Fahad-Alam-Jamal/Flyrank_ML_Internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fahad-Alam-Jamal/Flyrank_ML_Internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This notebook trains a simple, interpretable model for the refresh lane and compares it to the Week-4 rule baseline on the same held-out split.

> The goal is not to chase the highest score at any cost. The goal is to show whether a learned model is meaningfully better than the simple rule, and to explain where it still fails.

## 0. Setup (Colab or local)
On Colab this clones the repo and installs requirements. Locally it just moves to the repo root.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter
Starter data found. You're ready.


In [2]:
import os
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, precision_score, recall_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


def find_repo_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    while True:
        if (current / "data" / "raw" / "content_refresh_anonymized.csv").exists():
            return current
        if current == current.parent:
            return current
        current = current.parent


repo_root = find_repo_root()
if repo_root != Path.cwd():
    os.chdir(repo_root)

feature_path = repo_root / "data" / "processed" / "refresh_feature_vector.csv"
baseline_path = repo_root / "data" / "processed" / "baseline_refresh_queue.csv"

if not feature_path.exists():
    subprocess.run([sys.executable, "scripts/01_prepare_features.py"], cwd=repo_root, check=True)
if not baseline_path.exists():
    subprocess.run([sys.executable, "scripts/02_baseline_score.py"], cwd=repo_root, check=True)

features = pd.read_csv(feature_path)
baseline_queue = pd.read_csv(baseline_path)

print(f"Loaded {len(features):,} prepared rows")
print(f"Loaded baseline queue with {len(baseline_queue):,} rows")

Loaded 30,000 prepared rows
Loaded baseline queue with 30,000 rows


## 1. Method choice and why

I chose logistic regression because the task is a binary decision problem with an observed label and a need for transparent, auditable ranking. A simple logistic model is easier to inspect than a tree ensemble, and it gives calibrated probabilities that can be used for a ranked review queue. I also prefer it here because the lane is about decision support rather than squeezing out a few points with a much more opaque model.

In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder


numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "log_impressions_90d",
    "log_clicks_90d",
    "log_sessions_90d",
    "log_ai_sessions_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
]

categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "impression_tier",
    "position_tier",
]

model_features = features[numeric_features + categorical_features].copy()
model_features[numeric_features] = model_features[numeric_features].apply(pd.to_numeric, errors="coerce")
model_features[numeric_features] = model_features[numeric_features].fillna(0)
model_features[categorical_features] = model_features[categorical_features].fillna("unknown").astype(str)

target = features["is_declining_label"].astype(int)

preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="constant", fill_value=0))]), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

model = Pipeline(
    steps=[
        ("preprocess", preprocess),
        ("classifier", LogisticRegression(class_weight="balanced", max_iter=2000, random_state=42)),
    ]
)

print("Prepared model input matrix with", model_features.shape[1], "columns")

Prepared model input matrix with 26 columns


## 2. Split design

I use a client-aware holdout split rather than a random row-level split. That is more honest for this problem because the data are clustered by client behavior and a random split could leak near-duplicate patterns across the train and test sets. The test set contains roughly 20% of clients, and I keep the split fixed with a seed so the comparison is reproducible.

In [4]:
RANDOM_STATE = 42

client_series = features["client_id"].fillna("unknown").astype(str)
clients = client_series.drop_duplicates().to_numpy()
rng = np.random.default_rng(RANDOM_STATE)
shuffled_clients = rng.permutation(clients)
test_client_count = max(1, int(round(len(shuffled_clients) * 0.2)))

# Keep trying until both classes appear in the held-out client set.
for attempt in range(50):
    test_clients = set(shuffled_clients[:test_client_count])
    test_mask = client_series.isin(test_clients).to_numpy()
    train_mask = ~test_mask
    if target.iloc[train_mask].nunique() == 2 and target.iloc[test_mask].nunique() == 2:
        break
    test_client_count = min(len(shuffled_clients) - 1, test_client_count + 1)

if not (target.iloc[train_mask].nunique() == 2 and target.iloc[test_mask].nunique() == 2):
    raise RuntimeError("Could not build a client-holdout split with both classes in train and test")

train_features = model_features.iloc[train_mask]
test_features = model_features.iloc[test_mask]
train_target = target.iloc[train_mask]
test_target = target.iloc[test_mask]

print("Train rows:", len(train_features))
print("Test rows:", len(test_features))
print("Held-out clients:", len(test_clients))
print("Test positive rate:", round(float(test_target.mean()), 3))

Train rows: 27675
Test rows: 2325
Held-out clients: 6
Test positive rate: 0.391


## 3. Train + compare vs my baseline

I will fit the logistic regression on the train portion and rank the test rows by predicted probability. I then compare that ranking against the Week-4 rule baseline using the same test rows and the same ranking metric: precision at 20, 50, and 100.

In [5]:

def precision_at_k(y_true, scores, k):
    frame = pd.DataFrame({"y": list(y_true), "score": list(scores)})
    if frame.empty:
        return 0.0
    top = frame.sort_values("score", ascending=False).head(min(k, len(frame)))
    return float(top["y"].mean()) if len(top) else 0.0


model.fit(train_features, train_target)
model_probabilities = model.predict_proba(test_features)[:, 1]

baseline_lookup = baseline_queue.set_index("content_id")["baseline_refresh_score"]
base_test_scores = features.loc[test_mask, "content_id"].map(baseline_lookup).fillna(0).to_numpy()

metrics = {
    "base_rate": float(test_target.mean()),
    "baseline_precision_at_20": precision_at_k(test_target, base_test_scores, 20),
    "baseline_precision_at_50": precision_at_k(test_target, base_test_scores, 50),
    "baseline_precision_at_100": precision_at_k(test_target, base_test_scores, 100),
    "model_precision_at_20": precision_at_k(test_target, model_probabilities, 20),
    "model_precision_at_50": precision_at_k(test_target, model_probabilities, 50),
    "model_precision_at_100": precision_at_k(test_target, model_probabilities, 100),
    "baseline_roc_auc": float(roc_auc_score(test_target, base_test_scores)),
    "model_roc_auc": float(roc_auc_score(test_target, model_probabilities)),
    "baseline_average_precision": float(average_precision_score(test_target, base_test_scores)),
    "model_average_precision": float(average_precision_score(test_target, model_probabilities)),
}

summary = pd.DataFrame(
    {
        "metric": [
            "precision_at_20",
            "precision_at_50",
            "precision_at_100",
            "roc_auc",
            "average_precision",
        ],
        "baseline": [
            metrics["baseline_precision_at_20"],
            metrics["baseline_precision_at_50"],
            metrics["baseline_precision_at_100"],
            metrics["baseline_roc_auc"],
            metrics["baseline_average_precision"],
        ],
        "model": [
            metrics["model_precision_at_20"],
            metrics["model_precision_at_50"],
            metrics["model_precision_at_100"],
            metrics["model_roc_auc"],
            metrics["model_average_precision"],
        ],
    }
)
summary["delta"] = summary["model"] - summary["baseline"]
summary.round(3)

print("Comparison table")
print(summary.to_string(index=False))
print("\nBase rate in test set:", round(metrics["base_rate"], 3))

comparison_path = repo_root / "work" / "outputs" / "model_comparison_metrics.csv"
comparison_path.parent.mkdir(parents=True, exist_ok=True)
summary.to_csv(comparison_path, index=False)
print(f"Wrote comparison table to {comparison_path}")

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Comparison table
           metric  baseline    model    delta
  precision_at_20  0.150000 0.700000 0.550000
  precision_at_50  0.240000 0.700000 0.460000
 precision_at_100  0.360000 0.610000 0.250000
          roc_auc  0.626892 0.688257 0.061365
average_precision  0.467607 0.552905 0.085299

Base rate in test set: 0.391
Wrote comparison table to /content/flyrank-ml-internship-starter/work/outputs/model_comparison_metrics.csv


## 4. Errors and interpretation

The logistic model improves ranking quality on the held-out set, but it still makes clear mistakes. The most useful thing to inspect is not just the aggregate number; it is the pattern of the errors. In this lane, the model is most likely to over-rank pages that look active by volume but are not clearly stale or declining, and to miss pages where the signal is more subtle and the traffic is still modest.

In [6]:
# Build a small error table for interpretation.

test_frame = features.loc[test_mask].copy()
test_frame["baseline_score"] = base_test_scores
test_frame["model_probability"] = model_probabilities
test_frame["model_prediction"] = (model_probabilities >= 0.5).astype(int)
test_frame["true_label"] = test_target.to_numpy()
test_frame["error_type"] = np.where(
    (test_frame["model_prediction"] == 1) & (test_frame["true_label"] == 0),
    "false_positive",
    np.where((test_frame["model_prediction"] == 0) & (test_frame["true_label"] == 1), "false_negative", "correct"),
)

false_positive_examples = test_frame.loc[test_frame["error_type"] == "false_positive"].sort_values(
    "model_probability", ascending=False
).head(3)
false_negative_examples = test_frame.loc[test_frame["error_type"] == "false_negative"].sort_values(
    "model_probability", ascending=True
).head(3)

print("False positives (top 3 by model confidence):")
print(false_positive_examples[["content_id", "impressions_90d", "days_since_last_update", "avg_position", "ctr", "model_probability", "baseline_score"]].to_string(index=False))
print("\nFalse negatives (top 3 by low model confidence):")
print(false_negative_examples[["content_id", "impressions_90d", "days_since_last_update", "avg_position", "ctr", "model_probability", "baseline_score"]].to_string(index=False))

# Permutation importance for a human-readable feature view.
perm = permutation_importance(
    model,
    train_features,
    train_target,
    n_repeats=20,
    random_state=42,
    scoring="average_precision",
)
importance_frame = pd.DataFrame({
    "feature": train_features.columns,
    "importance": perm.importances_mean,
}).sort_values("importance", ascending=False).head(10)

print("\nTop permutation-importance features:")
print(importance_frame.to_string(index=False))

False positives (top 3 by model confidence):
          content_id  impressions_90d  days_since_last_update  avg_position  ctr  model_probability  baseline_score
content_9f9d01b5fedd            30953                      20           8.4 0.06           0.817814        0.705604
content_8fdbff16a886             3863                      20          12.0 0.00           0.807372        0.559004
content_da806b9f243e             2268                      20           5.8 0.18           0.805213        0.531936

False negatives (top 3 by low model confidence):
          content_id  impressions_90d  days_since_last_update  avg_position    ctr  model_probability  baseline_score
content_a8cee66e4788                1                      20           2.0 100.00       2.755160e-08        0.113147
content_e0e412fa582f             1719                       8           8.9   0.58       2.537080e-07        0.421382
content_7ff6ded89d4a             1083                      20          13.4   0.28     

The model leans most heavily on traffic and freshness-related features, which is sensible for a refresh lane. The top signals are consistent with the problem: pages that are both visible and aging are likely to be worth review, while very high-volume pages with no obvious freshness problem can still look overconfident to the model. That is why the strongest error cases are not random; they are concentrated in pages where the evidence is mixed rather than clearly positive or negative.

## Self-check

Before I submit, I confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] I kept the work under the repo path [work/notebooks](work/notebooks)